# 01 -- Argparse avance

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- creer des CLI robustes avec `argparse` (stdlib) ;
- utiliser positionnels, options, flags, nargs ;
- declarer des sous-commandes avec `subparsers` ;
- valider et convertir les entrees avec `type=` ;
- utiliser `choices`, `required`, `default` ;
- personnaliser l'aide et les erreurs ;
- lire des arguments depuis un fichier ;
- tester un parser.

## Prerequis -- ce que vous devez maitriser

- fonctions typees ;
- la notion de CLI et de code de retour.

Ce que ce notebook **ne couvrira pas** :

- Click (voir `02_Click.ipynb`) ;
- Typer (voir `03_Typer.ipynb`) ;
- les CLI interactives (Rich, Textual).

## Plan

1. Premier parser
2. Arguments positionnels
3. Options
4. Flags booleens
5. Types et conversions
6. Nargs
7. Choices et validation
8. Sous-commandes
9. Parents et herirage
10. Fichiers d'arguments
11. Personnalisation
12. Tester un parser
13. Synthese
14. Exercices

---

## 1. Premier parser

In [ ]:
import argparse

parser = argparse.ArgumentParser(description='Demo CLI')
parser.add_argument('name', help='Votre nom')
args = parser.parse_args(['Alice'])
print(args.name)

---

## 2. Arguments positionnels

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument('src')
parser.add_argument('dst')
args = parser.parse_args(['fichier.txt', '/backup/'])
print(args.src, '->', args.dst)

In [ ]:
# Plusieurs positionnels avec nargs='+'
parser = argparse.ArgumentParser()
parser.add_argument('files', nargs='+')
args = parser.parse_args(['a.txt', 'b.txt', 'c.txt'])
args.files

---

## 3. Options

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument('--output', '-o', default='out.txt')
parser.add_argument('--verbose', '-v', action='count', default=0)
args = parser.parse_args(['-o', 'result.txt', '-vvv'])
print(args.output, args.verbose)

---

## 4. Flags booleens

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument('--debug', action='store_true')
parser.add_argument('--no-cache', action='store_false', dest='cache')
args = parser.parse_args(['--debug', '--no-cache'])
print(args.debug, args.cache)

Python 3.9+ : `action='store_true'` / `'store_false'` / `BooleanOptionalAction`.

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument('--color', action=argparse.BooleanOptionalAction, default=True)
parser.parse_args(['--no-color']).color

---

## 5. Types et conversions

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument('--count', type=int, default=1)
parser.add_argument('--rate', type=float, default=1.0)
args = parser.parse_args(['--count', '5', '--rate', '0.25'])
print(args.count, args.rate)

In [ ]:
# type custom
def positive_int(value: str) -> int:
    v = int(value)
    if v <= 0:
        raise argparse.ArgumentTypeError(f'doit etre positif, recu {v}')
    return v

parser = argparse.ArgumentParser()
parser.add_argument('-n', type=positive_int)
args = parser.parse_args(['-n', '5'])
args.n

In [ ]:
# type=pathlib.Path pour convertir automatiquement
import pathlib
parser = argparse.ArgumentParser()
parser.add_argument('--config', type=pathlib.Path)
args = parser.parse_args(['--config', '/etc/myapp.conf'])
type(args.config), args.config

---

## 6. Nargs

### Variantes de nargs

| Valeur | Effet |
|---|---|
| `N` (int) | exactement N valeurs |
| `'?'` | 0 ou 1 valeur |
| `'*'` | 0 ou plusieurs |
| `'+'` | 1 ou plusieurs |
| `argparse.REMAINDER` | le reste brut |

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument('--coords', nargs=2, type=float)
args = parser.parse_args(['--coords', '48.85', '2.35'])
args.coords

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument('files', nargs='*')
args = parser.parse_args([])
args.files

---

## 7. Choices et validation

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument('--level', choices=['debug', 'info', 'warning', 'error'])
args = parser.parse_args(['--level', 'info'])
args.level

In [ ]:
# Plage via choices=range(...)
parser = argparse.ArgumentParser()
parser.add_argument('--port', type=int, choices=range(1024, 65536))
print('Choices sur 64k ports')

---

## 8. Sous-commandes

In [ ]:
parser = argparse.ArgumentParser(prog='git')
sub = parser.add_subparsers(dest='cmd', required=True)

add = sub.add_parser('add', help='ajouter un fichier')
add.add_argument('files', nargs='+')

commit = sub.add_parser('commit', help='creer un commit')
commit.add_argument('-m', '--message', required=True)

args = parser.parse_args(['commit', '-m', 'initial'])
print(args.cmd, args.message)

In [ ]:
args = parser.parse_args(['add', 'a.py', 'b.py'])
print(args.cmd, args.files)

---

## 9. Parents et heritage

In [ ]:
common = argparse.ArgumentParser(add_help=False)
common.add_argument('--verbose', '-v', action='store_true')
common.add_argument('--config', type=pathlib.Path, default=pathlib.Path('default.conf'))

parser = argparse.ArgumentParser(parents=[common])
parser.add_argument('target')

args = parser.parse_args(['-v', 'prod'])
print(args.verbose, args.target, args.config)

---

## 10. Fichiers d'arguments

In [ ]:
import tempfile, pathlib
tmp = pathlib.Path(tempfile.gettempdir()) / 'args.txt'
tmp.write_text('--output\nresult.txt\n--verbose\n')

parser = argparse.ArgumentParser(fromfile_prefix_chars='@')
parser.add_argument('--output')
parser.add_argument('--verbose', action='store_true')

args = parser.parse_args(['@' + str(tmp)])
print(args)

---

## 11. Personnalisation

In [ ]:
# Aide personnalisee
parser = argparse.ArgumentParser(
    prog='myapp',
    description='Un outil super cool',
    epilog='Bugs : https://example.com/bugs',
    formatter_class=argparse.ArgumentDefaultsHelpFormatter,
)
parser.add_argument('--retries', type=int, default=3, help='nombre de tentatives')
parser.format_help()[:200]

---

## 12. Tester un parser

In [ ]:
def build_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser()
    p.add_argument('--count', type=int, default=1)
    return p

def test_default():
    p = build_parser()
    assert p.parse_args([]).count == 1

def test_explicit():
    p = build_parser()
    assert p.parse_args(['--count', '5']).count == 5

test_default(); test_explicit()
print('OK')

---

## Synthese

| Element | argparse |
|---|---|
| Parser | `argparse.ArgumentParser(description=...)` |
| Positionnel | `parser.add_argument('name')` |
| Option | `parser.add_argument('--opt', '-o')` |
| Flag | `action='store_true'` / `BooleanOptionalAction` |
| Type | `type=int`, `type=pathlib.Path`, `type=func` |
| Validation | `choices=[...]`, `type=...` |
| Multi-valeurs | `nargs=2`, `'+'`, `'*'` |
| Sous-cmd | `subparsers = parser.add_subparsers(dest='cmd')` |
| Fichier args | `fromfile_prefix_chars='@'` |

### Regles a retenir

- **argparse = stdlib**, pas de dep externe.
- **`dest=`** pour renommer le champ.
- **`required=True`** pour forcer une option.
- **`subparsers` + `dest`** pour dispatcher.
- **Parents** pour factoriser des args communs.

---

## Exercices

### Exercice 1 (facile) -- Copie de fichier

Creez un parser qui accepte `src` et `dst` en positionnels et une option `-v/--verbose`.

In [ ]:
import argparse

# TODO

<details>
<summary>Voir la correction</summary>

```python
import argparse

def build() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(description='Copy files')
    p.add_argument('src')
    p.add_argument('dst')
    p.add_argument('-v', '--verbose', action='store_true')
    return p

print(build().parse_args(['a.txt', 'b.txt', '-v']))
```

</details>

### Exercice 2 (facile) -- Niveau de log

Parser qui accepte `--log-level` parmi `DEBUG, INFO, WARNING, ERROR` (defaut INFO).

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import argparse
p = argparse.ArgumentParser()
p.add_argument('--log-level', choices=['DEBUG', 'INFO', 'WARNING', 'ERROR'], default='INFO')
print(p.parse_args(['--log-level', 'DEBUG']).log_level)
```

</details>

### Exercice 3 (moyen) -- CLI avec sous-commandes

CLI `todo` avec sous-commandes `add <text>`, `list`, `done <id>`.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import argparse

p = argparse.ArgumentParser(prog='todo')
sub = p.add_subparsers(dest='cmd', required=True)
sub.add_parser('list')
add = sub.add_parser('add'); add.add_argument('text')
done = sub.add_parser('done'); done.add_argument('id', type=int)

print(p.parse_args(['add', 'acheter du pain']))
print(p.parse_args(['done', '3']))
```

</details>

### Exercice 4 (moyen) -- Type custom

Creez un type `port` qui accepte un entier entre 1 et 65535, sinon `ArgumentTypeError`.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import argparse

def port(value: str) -> int:
    v = int(value)
    if not 1 <= v <= 65535:
        raise argparse.ArgumentTypeError(f'port hors limite: {v}')
    return v

p = argparse.ArgumentParser()
p.add_argument('-p', '--port', type=port, default=8000)
print(p.parse_args(['-p', '8080']).port)
```

</details>

### Exercice 5 (difficile) -- CLI factorise

Deux sous-commandes (`build` et `deploy`) qui partagent des options communes (`--env`, `--verbose`) via un parser parent.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import argparse

common = argparse.ArgumentParser(add_help=False)
common.add_argument('--env', choices=['dev', 'prod'], default='dev')
common.add_argument('-v', '--verbose', action='store_true')

p = argparse.ArgumentParser(prog='app')
sub = p.add_subparsers(dest='cmd', required=True)
build = sub.add_parser('build', parents=[common])
build.add_argument('--target', default='all')
deploy = sub.add_parser('deploy', parents=[common])
deploy.add_argument('--host', required=True)

print(p.parse_args(['build', '--env', 'prod', '-v']))
print(p.parse_args(['deploy', '--host', 'server1']))
```

</details>

---

## Ressources externes

- [argparse - stdlib](https://docs.python.org/3/library/argparse.html)
- [argparse tutorial](https://docs.python.org/3/howto/argparse.html)
- [PEP 389 - argparse](https://peps.python.org/pep-0389/)